---
## **[실습 프로젝트]**

- 옵션 2, 옵션 3에서 적용한 멀티모달 RAG 시스템을 활용하여 증권사 분석보고서에 대한 질문에 답변하는 시스템을 구축합니다.
- 멀티모달 컨텍스트를 구성하는 다양한 방법과 차이점에 대해 비교합니다.

# Docling 기반 멀티모달 RAG 시스템 플로우

```mermaid
flowchart TD
    Start([멀티모달 RAG 시스템 시작]) --> LoadPDF

    %% 데이터 로딩 단계
    subgraph DataLoading["데이터 로딩 및 파싱"]
        LoadPDF[PDF 파일 로드]
        LoadPDF --> Converter{DocumentConverter 초기화}
        Converter --> |"PdfPipelineOptions<br/>images_scale=2.0<br/>generate_page_images=True"| ExtractAll[문서 변환 및 추출]
        ExtractAll --> Chunker{HybridChunker}
        Chunker --> |"tokenizer='BAAI/bge-m3'<br/>max_tokens=8000"| ChunkResult[청킹 완료]
    end

    %% 1단계: 데이터 준비
    subgraph Step1["1단계: 데이터 준비"]
        ChunkResult --> SplitData[타입별 데이터 분리]
        SplitData --> TextChunks[텍스트 청크 수집<br/>metadata: source, chunk_index, pages]
        SplitData --> Tables[테이블 수집<br/>markdown 형식<br/>metadata: source, page_no]
        SplitData --> Images[페이지 이미지 수집<br/>base64 형식<br/>metadata: source, page_no, width, height]
    end

    %% 2-4단계: 요약 생성
    subgraph Step2_4["2-4단계: 요약 생성"]
        TextChunks --> TextSumm{텍스트 요약 생성}
        TextSumm --> |"ChatOpenAI<br/>model='gpt-4o-mini'<br/>핵심 내용 및 주요 수치 요약"| TextSummResult[텍스트 요약 완료]

        Tables --> TableSumm{테이블 요약 생성}
        TableSumm --> |"ChatOpenAI<br/>model='gpt-4o-mini'<br/>주요 수치 및 패턴 분석"| TableSummResult[테이블 요약 완료]

        Images --> ImgSumm{이미지 요약 생성}
        ImgSumm --> |"ChatOpenAI (Multimodal)<br/>model='gpt-4o-mini'<br/>이미지 분석 및 비즈니스 의미 추출"| ImgSummResult[이미지 요약 완료]
    end

    %% 5단계: MultiVectorRetriever 초기화
    subgraph Step5["5단계: MultiVectorRetriever 초기화"]
        TextSummResult --> InitRetriever
        TableSummResult --> InitRetriever
        ImgSummResult --> InitRetriever
        InitRetriever[Retriever 초기화] --> VectorStore{Chroma VectorStore}
        VectorStore --> |"collection='docling_multimodal_rag'<br/>embedding='text-embedding-3-small'<br/>persist_directory='./docling_chroma_db'"| VS_Created[벡터스토어 생성]
        InitRetriever --> DocStore{InMemoryStore}
        DocStore --> DS_Created[문서스토어 생성]
        VS_Created --> MVRetriever[MultiVectorRetriever]
        DS_Created --> MVRetriever
    end

    %% 6-8단계: 문서 추가
    subgraph Step6_8["6-8단계: 문서 추가"]
        MVRetriever --> AddText[텍스트 추가]
        AddText --> |"요약 → VectorStore<br/>원본 텍스트 → DocStore"| TextAdded[텍스트 저장 완료]

        MVRetriever --> AddTable[테이블 추가]
        AddTable --> |"요약 → VectorStore<br/>원본 마크다운 → DocStore"| TableAdded[테이블 저장 완료]

        MVRetriever --> AddImage[이미지 추가]
        AddImage --> |"요약 → VectorStore<br/>원본 base64 → DocStore"| ImageAdded[이미지 저장 완료]

        TextAdded --> StoreReady[벡터스토어 구성 완료]
        TableAdded --> StoreReady
        ImageAdded --> StoreReady
    end

    %% 9-10단계: RAG 체인 구성
    subgraph Step9_10["9-10단계: RAG 체인 구성"]
        StoreReady --> PromptFunc[프롬프트 처리 함수 정의]
        PromptFunc --> |"docling_process_prompt()<br/>- 타입별 문서 분류<br/>- 텍스트 컨텍스트 구성<br/>- 이미지 추가"| FuncDefined[함수 정의 완료]

        FuncDefined --> BuildChain[RAG 체인 구성]
        BuildChain --> BasicChain["기본 체인<br/>context: retriever<br/>question: passthrough"]
        BuildChain --> SourceChain["소스 포함 체인<br/>response + context 반환"]

        BasicChain --> ChainReady[RAG 체인 준비 완료]
        SourceChain --> ChainReady
    end

    %% 11-14단계: 테스트 및 비교
    subgraph Step11_14["11-14단계: 테스트 질의"]
        ChainReady --> Query1{질문 1: 기본 질문}
        Query1 --> |"텍스트 검색<br/>실적 전망 질문"| Answer1[답변 생성]

        ChainReady --> Query2{질문 2: 소스 포함}
        Query2 --> |"재무 지표 질문<br/>메타데이터 포함 반환"| Answer2[답변 + 소스 반환]

        ChainReady --> Query3{질문 3: 이미지 포함}
        Query3 --> |"차트/그래프 트렌드<br/>멀티모달 처리"| Answer3[답변 + 이미지 표시]

        Answer1 --> Compare[옵션 2 vs 옵션 3 비교]
        Answer2 --> Compare
        Answer3 --> Compare
    end

    Compare --> End([시스템 완료])

    %% 스타일 정의
    classDef summaryStyle fill:#fff9c4,stroke:#f57f17,stroke-width:2px
    classDef storeStyle fill:#e1f5fe,stroke:#01579b,stroke-width:2px
    classDef chainStyle fill:#f3e5f5,stroke:#4a148c,stroke-width:2px
    classDef testStyle fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px

    class TextSumm,TableSumm,ImgSumm summaryStyle
    class VectorStore,DocStore,MVRetriever storeStyle
    class PromptFunc,BuildChain,BasicChain,SourceChain chainStyle
    class Query1,Query2,Query3,Compare testStyle
```

In [ ]:
# ========================================
# 1단계: 데이터 준비 - 텍스트 청크, 테이블, 이미지 분리
# ========================================

# Docling으로 추출한 데이터에서 타입별로 분리
docling_texts = []
docling_tables = []
docling_images = []

for file_path, doc_content in all_documents.items():
    file_name = os.path.basename(file_path)
    
    # 텍스트 청크 수집
    for chunk in doc_content['text_chunks']:
        docling_texts.append({
            'text': chunk['text'],
            'metadata': {
                'source': file_name,
                'chunk_index': chunk['index'],
                'token_count': chunk['token_count'],
                'pages': chunk['metadata'].get('pages', [])
            }
        })
    
    # 테이블 수집 (마크다운 형식)
    for table_key, table_info in doc_content['tables'].items():
        docling_tables.append({
            'markdown': table_info['markdown'],
            'metadata': {
                'source': file_name,
                'table_key': table_key,
                'page_no': table_info['page_no'],
                'table_index': table_info['table_index']
            }
        })
    
    # 페이지 이미지 수집 (base64)
    for page_key, image_info in doc_content['page_images'].items():
        # "data:image/png;base64," 프리픽스 제거
        base64_data = image_info['base64'].replace("data:image/png;base64,", "")
        docling_images.append({
            'base64': base64_data,
            'metadata': {
                'source': file_name,
                'page_key': page_key,
                'page_no': image_info['page_no'],
                'width': image_info['width'],
                'height': image_info['height']
            }
        })

print(f"텍스트 청크: {len(docling_texts)}개")
print(f"테이블: {len(docling_tables)}개")
print(f"페이지 이미지: {len(docling_images)}개")

In [ ]:
# 샘플 데이터 확인
print("=" * 60)
print("텍스트 청크 샘플:")
print(f"내용: {docling_texts[0]['text'][:150]}...")
print(f"메타데이터: {docling_texts[0]['metadata']}")

print("\n" + "=" * 60)
print("테이블 샘플:")
print(f"마크다운 (앞부분):\n{docling_tables[0]['markdown'][:200]}...")
print(f"메타데이터: {docling_tables[0]['metadata']}")

print("\n" + "=" * 60)
print("이미지 샘플:")
print(f"Base64 길이: {len(docling_images[0]['base64'])} 문자")
print(f"메타데이터: {docling_images[0]['metadata']}")

In [ ]:
# ========================================
# 2단계: 텍스트 요약 생성 (ChatGPT 활용)
# ========================================

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 텍스트 요약용 LLM 초기화
text_summarize_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 텍스트 요약 생성
docling_text_summaries = []

print("텍스트 청크 요약 생성 중...")
for i, text_data in enumerate(docling_texts[:5]):  # 테스트를 위해 처음 5개만 처리
    text_content = text_data['text']
    
    # 요약 프롬프트
    prompt = f"""
    다음 텍스트를 간결하게 요약해주세요. 핵심 내용과 주요 수치를 포함하세요.
    
    텍스트:
    {text_content}
    
    요약 (한국어):
    """
    
    # LLM 호출
    response = text_summarize_llm.invoke([HumanMessage(content=prompt)])
    summary = response.content
    
    docling_text_summaries.append(summary)
    
    if (i + 1) % 5 == 0:
        print(f"  진행: {i + 1}/{len(docling_texts[:5])}개 완료")

print(f"\n완료: 텍스트 요약 {len(docling_text_summaries)}개 생성")

In [ ]:
# ========================================
# 3단계: 테이블 요약 생성 (ChatGPT 활용)
# ========================================

# 테이블 요약 생성
docling_table_summaries = []

print("테이블 요약 생성 중...")
for i, table_data in enumerate(docling_tables[:5]):  # 테스트를 위해 처음 5개만 처리
    table_markdown = table_data['markdown']
    
    # 요약 프롬프트
    prompt = f"""
    다음 표를 분석하고 핵심 내용을 요약해주세요. 주요 수치와 패턴을 포함하세요.
    
    표:
    {table_markdown}
    
    요약 (한국어):
    """
    
    # LLM 호출
    response = text_summarize_llm.invoke([HumanMessage(content=prompt)])
    summary = response.content
    
    docling_table_summaries.append(summary)
    
    if (i + 1) % 5 == 0:
        print(f"  진행: {i + 1}/{len(docling_tables[:5])}개 완료")

print(f"\n완료: 테이블 요약 {len(docling_table_summaries)}개 생성")

In [ ]:
# ========================================
# 4단계: 이미지 요약 생성 (GPT-4V 활용)
# ========================================

# 이미지 요약용 멀티모달 LLM 초기화
image_summarize_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 이미지 요약 생성
docling_image_summaries = []

print("이미지 요약 생성 중...")
for i, image_data in enumerate(docling_images[:5]):  # 테스트를 위해 처음 5개만 처리
    base64_image = image_data['base64']
    
    # 이미지 요약 프롬프트
    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": """이 이미지를 분석하고 다음 내용을 포함하여 요약해주세요:
                1. 이미지의 주요 내용 (차트, 표, 텍스트 등)
                2. 핵심 데이터나 메시지
                3. 비즈니스 분석 관점에서의 의미
                
                요약 (한국어):"""
            },
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{base64_image}"}
            }
        ]
    )
    
    # LLM 호출
    response = image_summarize_llm.invoke([message])
    summary = response.content
    
    docling_image_summaries.append(summary)
    
    if (i + 1) % 5 == 0:
        print(f"  진행: {i + 1}/{len(docling_images[:5])}개 완료")

print(f"\n완료: 이미지 요약 {len(docling_image_summaries)}개 생성")

In [ ]:
# 요약 결과 확인
print("=" * 60)
print("생성된 요약 통계:")
print(f"  텍스트 요약: {len(docling_text_summaries)}개")
print(f"  테이블 요약: {len(docling_table_summaries)}개")
print(f"  이미지 요약: {len(docling_image_summaries)}개")

print("\n" + "=" * 60)
print("텍스트 요약 샘플:")
print(docling_text_summaries[0][:300] + "...")

print("\n" + "=" * 60)
print("테이블 요약 샘플:")
print(docling_table_summaries[0][:300] + "...")

print("\n" + "=" * 60)
print("이미지 요약 샘플:")
print(docling_image_summaries[0][:300] + "...")

In [ ]:
# ========================================
# 5단계: MultiVectorRetriever 초기화
# ========================================

import uuid
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.stores import InMemoryStore
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

# 벡터 저장소 생성 (요약을 임베딩하여 저장)
docling_vectorstore = Chroma(
    collection_name="docling_multimodal_rag", 
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./docling_chroma_db",
)

# 문서 저장소 생성 (원본 데이터 저장)
docling_store = InMemoryStore()
docling_id_key = "doc_id"

# MultiVectorRetriever 생성
docling_retriever = MultiVectorRetriever(
    vectorstore=docling_vectorstore,
    docstore=docling_store,
    id_key=docling_id_key,
)

print("MultiVectorRetriever 초기화 완료")
print(f"  벡터스토어 컬렉션: docling_multimodal_rag")
print(f"  임베딩 모델: text-embedding-3-small")
print(f"  저장 디렉토리: ./docling_chroma_db")

In [ ]:
# ========================================
# 6단계: 텍스트 청크 추가
# ========================================

# 텍스트 청크에 대한 고유 ID 생성
text_doc_ids = [str(uuid.uuid4()) for _ in docling_text_summaries]

# 요약을 벡터스토어에 저장할 Document 생성
text_summary_docs = [
    Document(
        page_content=summary, 
        metadata={
            docling_id_key: text_doc_ids[i], 
            "source": docling_texts[i]['metadata']['source'],
            "type": "text",
            "chunk_index": docling_texts[i]['metadata']['chunk_index']
        }
    )
    for i, summary in enumerate(docling_text_summaries)
]

# 원본 텍스트를 docstore에 저장할 Document 생성
text_original_docs = [
    Document(
        page_content=docling_texts[i]['text'], 
        metadata={
            docling_id_key: text_doc_ids[i], 
            "source": docling_texts[i]['metadata']['source'],
            "type": "text",
            "chunk_index": docling_texts[i]['metadata']['chunk_index']
        }
    )
    for i in range(len(docling_text_summaries))
]

# 벡터스토어에 요약 추가
docling_retriever.vectorstore.add_documents(text_summary_docs)

# docstore에 원본 텍스트 추가
docling_retriever.docstore.mset(list(zip(text_doc_ids, text_original_docs)))

print(f"텍스트 청크 추가 완료: {len(text_summary_docs)}개")
print(f"  벡터스토어: 요약 {len(text_summary_docs)}개")
print(f"  docstore: 원본 텍스트 {len(text_original_docs)}개")

In [ ]:
# ========================================
# 7단계: 테이블 추가
# ========================================

# 테이블에 대한 고유 ID 생성
table_doc_ids = [str(uuid.uuid4()) for _ in docling_table_summaries]

# 요약을 벡터스토어에 저장할 Document 생성
table_summary_docs = [
    Document(
        page_content=summary, 
        metadata={
            docling_id_key: table_doc_ids[i], 
            "source": docling_tables[i]['metadata']['source'],
            "type": "table",
            "page_no": docling_tables[i]['metadata']['page_no']
        }
    )
    for i, summary in enumerate(docling_table_summaries)
]

# 원본 테이블(마크다운)을 docstore에 저장할 Document 생성
table_original_docs = [
    Document(
        page_content=docling_tables[i]['markdown'], 
        metadata={
            docling_id_key: table_doc_ids[i], 
            "source": docling_tables[i]['metadata']['source'],
            "type": "table",
            "page_no": docling_tables[i]['metadata']['page_no']
        }
    )
    for i in range(len(docling_table_summaries))
]

# 벡터스토어에 요약 추가
docling_retriever.vectorstore.add_documents(table_summary_docs)

# docstore에 원본 테이블 추가
docling_retriever.docstore.mset(list(zip(table_doc_ids, table_original_docs)))

print(f"테이블 추가 완료: {len(table_summary_docs)}개")
print(f"  벡터스토어: 요약 {len(table_summary_docs)}개")
print(f"  docstore: 원본 테이블 {len(table_original_docs)}개")

In [ ]:
# ========================================
# 8단계: 이미지 추가
# ========================================

# 이미지에 대한 고유 ID 생성
image_doc_ids = [str(uuid.uuid4()) for _ in docling_image_summaries]

# 요약을 벡터스토어에 저장할 Document 생성
image_summary_docs = [
    Document(
        page_content=summary, 
        metadata={
            docling_id_key: image_doc_ids[i], 
            "source": docling_images[i]['metadata']['source'],
            "type": "image",
            "page_no": docling_images[i]['metadata']['page_no']
        }
    )
    for i, summary in enumerate(docling_image_summaries)
]

# 원본 이미지(base64)를 docstore에 저장할 Document 생성
image_original_docs = [
    Document(
        page_content=docling_images[i]['base64'], 
        metadata={
            docling_id_key: image_doc_ids[i], 
            "source": docling_images[i]['metadata']['source'],
            "type": "image",
            "page_no": docling_images[i]['metadata']['page_no']
        }
    )
    for i in range(len(docling_image_summaries))
]

# 벡터스토어에 요약 추가
docling_retriever.vectorstore.add_documents(image_summary_docs)

# docstore에 원본 이미지 추가
docling_retriever.docstore.mset(list(zip(image_doc_ids, image_original_docs)))

print(f"이미지 추가 완료: {len(image_summary_docs)}개")
print(f"  벡터스토어: 요약 {len(image_summary_docs)}개")
print(f"  docstore: 원본 이미지(base64) {len(image_original_docs)}개")

In [ ]:
# 벡터스토어 구성 완료 확인
print("=" * 60)
print("벡터스토어 구성 완료")
print(f"전체 문서 수:")
print(f"  텍스트: {len(text_summary_docs)}개")
print(f"  테이블: {len(table_summary_docs)}개")
print(f"  이미지: {len(image_summary_docs)}개")
print(f"  총합: {len(text_summary_docs) + len(table_summary_docs) + len(image_summary_docs)}개")

In [ ]:
# 검색 테스트
test_query = "삼성전기의 실적 전망은?"
retrieved_docs = docling_retriever.invoke(test_query)

print(f"검색 쿼리: {test_query}")
print(f"검색된 문서 수: {len(retrieved_docs)}개")
print("\n" + "=" * 60)

for i, doc in enumerate(retrieved_docs):
    print(f"\n문서 {i+1}:")
    print(f"  타입: {doc.metadata.get('type', 'unknown')}")
    print(f"  소스: {doc.metadata.get('source', 'unknown')}")
    print(f"  내용 (앞부분): {doc.page_content[:200]}...")
    print("-" * 60)

In [ ]:
# ========================================
# 9단계: RAG 체인 구성 - 프롬프트 처리 함수
# ========================================

from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

def docling_process_prompt(kwargs):
    """검색된 문서를 타입별로 분류하고 프롬프트 구성"""
    docs = kwargs["context"]
    user_question = kwargs["question"]
    
    # 타입별로 문서 분류
    text_docs = []
    table_docs = []
    image_docs = []
    
    for doc in docs:
        doc_type = doc.metadata.get('type', 'unknown')
        if doc_type == 'text':
            text_docs.append(doc.page_content)
        elif doc_type == 'table':
            table_docs.append(doc.page_content)
        elif doc_type == 'image':
            image_docs.append(doc.page_content)
    
    print(f"검색된 문서:")
    print(f"  텍스트: {len(text_docs)}개")
    print(f"  테이블: {len(table_docs)}개")
    print(f"  이미지: {len(image_docs)}개")
    print("-" * 60)
    
    # 텍스트 컨텍스트 구성
    context_text = ""
    
    if text_docs:
        context_text += "\n[텍스트 정보]\n"
        for i, text in enumerate(text_docs):
            context_text += f"\n{i+1}. {text}\n"
    
    if table_docs:
        context_text += "\n[테이블 정보]\n"
        for i, table in enumerate(table_docs):
            context_text += f"\n테이블 {i+1}:\n{table}\n"
    
    # 프롬프트 템플릿
    prompt_text = f"""
    제공된 컨텍스트를 기반으로 질문에 답변하세요. 비즈니스 분석 관점에서 답변해주세요.
    
    숫자 데이터나 통계를 제시할 때는 컨텍스트에서 해당 수치를 뒷받침하는 구체적인 근거를 인용하세요.
    컨텍스트에서 지원되지 않는 가정이나 일반화는 피하세요.
    
    [컨텍스트]
    {context_text}
    
    [질문]
    {user_question}
    
    [답변 (한국어)]
    """
    
    # 프롬프트 콘텐츠 구성
    prompt_content = [{"type": "text", "text": prompt_text}]
    
    # 이미지가 있으면 추가
    if image_docs:
        for image_base64 in image_docs:
            prompt_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{image_base64}"}
            })
    
    return ChatPromptTemplate.from_messages([
        HumanMessage(content=prompt_content)
    ])

print("프롬프트 처리 함수 정의 완료")

In [ ]:
# ========================================
# 10단계: RAG 체인 구성
# ========================================

# 멀티모달 LLM 초기화
docling_rag_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 기본 RAG 체인
docling_rag_chain = (
    {
        "context": docling_retriever,
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(docling_process_prompt)
    | docling_rag_llm
    | StrOutputParser()
)

# 소스 포함 RAG 체인
docling_rag_chain_with_sources = (
    {
        "context": docling_retriever,
        "question": RunnablePassthrough(),
    }
    | RunnablePassthrough().assign(
        response=(
            RunnableLambda(docling_process_prompt)
            | docling_rag_llm
            | StrOutputParser()
        )
    )
)

print("RAG 체인 구성 완료")
print("  - docling_rag_chain: 기본 체인 (답변만 반환)")
print("  - docling_rag_chain_with_sources: 소스 포함 체인 (답변 + 검색된 문서)")

In [ ]:
# ========================================
# 11단계: 테스트 질의 1 - 기본 질문 (텍스트 검색)
# ========================================

query1 = "삼성전기의 2024년 실적 전망은 어떻게 되나요?"

print(f"질문: {query1}")
print("=" * 60)
print("\n[답변]")
response1 = docling_rag_chain.invoke(query1)
print(response1)

In [ ]:
# ========================================
# 12단계: 테스트 질의 2 - 소스 포함 (컨텍스트 확인)
# ========================================

query2 = "삼성전기의 주요 재무 지표는?"

print(f"질문: {query2}")
print("=" * 60)
result2 = docling_rag_chain_with_sources.invoke(query2)

print("\n[답변]")
print(result2['response'])

print("\n" + "=" * 60)
print("[검색된 문서 메타데이터]")
for i, doc in enumerate(result2['context']):
    print(f"\n{i+1}. 타입: {doc.metadata.get('type')}, 소스: {doc.metadata.get('source')}")

In [ ]:
# ========================================
# 13단계: 테스트 질의 3 - 이미지 포함 질문
# ========================================

query3 = "분석 보고서의 차트나 그래프에서 보이는 주요 트렌드는?"

print(f"질문: {query3}")
print("=" * 60)
result3 = docling_rag_chain_with_sources.invoke(query3)

print("\n[답변]")
print(result3['response'])

print("\n" + "=" * 60)
print("[검색된 문서 타입 분포]")
types_count = {}
for doc in result3['context']:
    doc_type = doc.metadata.get('type', 'unknown')
    types_count[doc_type] = types_count.get(doc_type, 0) + 1

for doc_type, count in types_count.items():
    print(f"  {doc_type}: {count}개")

In [ ]:
# 검색된 이미지 확인 (있는 경우)
image_docs = [doc for doc in result3['context'] if doc.metadata.get('type') == 'image']

if image_docs:
    print(f"검색된 이미지 수: {len(image_docs)}개")
    print("\n첫 번째 이미지 표시:")
    plt_img_base64(image_docs[0].page_content)
else:
    print("검색된 이미지가 없습니다.")